In [1]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/elastic/lib/python3.10/site-packages/IPython/core/extensions.py:151: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_14.pickle

In [4]:
%%RecordEvent
%%time
### cell 14 ###

# Compute exactly the same "data_sub" as the original, but entirely on the GPU:
data_sub = (
    df.dropna(subset=["year_added"])           # drop any rows with missing year_added (value_counts would skip these)
      .groupby(["type", "year_added"])       # GPU groupby on type & year
      .size()                                   # count entries per (type, year)
      .unstack(level="year_added")            # pivot years into columns
      .fillna(0)                                # fill missing type–year combos with 0
      .loc[["TV Show", "Movie"]]            # ensure the row order matches the original
      .cumsum(axis=0)                           # cumulative sum down the two rows (for stacking)
      .T                                        # transpose → index=year, columns=type
)


CPU times: user 257 ms, sys: 92.3 ms, total: 349 ms
Wall time: 353 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten/o4_mini_high_small/checkpoints/post_cell_14_try_2.pickle